In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
SHA2_BITS = 256

In [0]:
%run ./../common/utilities

In [0]:
dbutils.widgets.text("catalog", "abcgroup", "Catalog")

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
crm_customers = f"{catalog}.{silver_schema}.crm_customers"
erp_customers = f"{catalog}.{silver_schema}.erp_customers"
erp_customer_location = f"{catalog}.{silver_schema}.erp_customer_location"


In [0]:
df = (
    spark.table(crm_customers).alias("ci")
    .join(
        spark.table(erp_customers).alias("ca"),
        F.col("ci.customer_number") == F.col("ca.customer_number"),
        "left"
    )
    .join(
        spark.table(erp_customer_location).alias("la"),
        F.col("ci.customer_number") == F.col("la.customer_number"),
        "left"
    )
    .select(
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("ci.customer_id"), F.lit("")),
                F.coalesce(F.col("ci.customer_number"), F.lit(""))
    
            ),
            SHA2_BITS
        ).alias("customer_key"),

        F.col("ci.customer_id"),
        F.col("ci.customer_number"),
        F.col("ci.first_name"),
        F.col("ci.last_name"),
        F.col("la.country"),
        F.col("ci.marital_status"),

        F.when(F.col("ci.gender") != "n/a", F.col("ci.gender"))
         .otherwise(F.coalesce(F.col("ca.gender"), F.lit("other")))
         .alias("gender"),

        F.col("ca.birth_date").alias("birthdate"),
        F.col("ci.created_date").alias("create_date")
    )
)

In [0]:
display(df.limit(10))

In [0]:
(
df.write
.mode("overwrite")
.format("delta")
.saveAsTable(f"{catalog}.{gold_schema}.dim_customers")
)

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {catalog}.{gold_schema}.dim_customers
    LIMIT 5
"""))